# MP × MP similarity matrix (agreement rate)

Builds the weighted adjacency matrix used by `louvain_algorithm.ipynb` from `votes.csv`.

**Similarity = agreement rate.** For two MPs $i$ and $j$, let $S_{ij}$ be the sessions where **both** cast a valid vote. Then

$$
A_{ij} = \frac{\#\{s \in S_{ij} : v_{is} = v_{js}\}}{|S_{ij}|}
$$

- **Absences (`vote == 3`) are never counted**: a session only enters $S_{ij}$ if neither MP was absent.
- `VALID_VOTES = (1, 2)` (For / Against) follows the group's binary convention used in `maverick_detection.ipynb`. Set it to `(1, 2, 4)` to treat abstention as a third voting option: then two MPs who both abstain agree, and an abstention disagrees with For and Against.
- Pairs with fewer than `MIN_SHARED` shared sessions get weight 0 (no edge), because their agreement rate is unreliable (same SE ≤ 0.05 argument as in `retrieve.ipynb`).
- **Unanimous sessions are dropped** (`DROP_UNANIMOUS = True`): if every MP who cast a valid vote voted the same way, the session adds +1 agreement to *every* pair regardless of party. That carries no information about who sides with whom and only compresses all rates towards 1. Unanimity is judged on `VALID_VOTES`, so with `(1, 2, 4)` a single abstention breaks it. Near-unanimous sessions (e.g. 170–5) are kept: the few dissenters are exactly the signal we want.
- MPs left **without any edge** (all their pairs below `MIN_SHARED`, typically short-serving substitutes) are removed, otherwise Louvain would report each of them as a one-MP community.
- The diagonal is 0, as `louvain_algorithm` expects.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

VALID_VOTES = (1, 2)        # (1, 2, 4) to count abstentions as a vote option
DROP_UNANIMOUS = True       # drop sessions where all valid votes are identical
MIN_SHARED = 100            # minimum shared sessions for a pair to get an edge
OUT_PATH = "similarity.npz"

df = pd.read_csv("votes.csv", parse_dates=["date"])
assert not df.duplicated(["session", "actor"]).any()
print(f"{len(df):,} rows | {df.actor.nunique()} MPs | {df.session.nunique()} sessions")

## MP × session vote matrix

One row per MP, one column per session. Entries are the vote code, or NaN when the MP was absent / not registered / cast a vote outside `VALID_VOTES`.

In [ ]:
valid = df[df["vote"].isin(VALID_VOTES)]

if DROP_UNANIMOUS:
    n_options = valid.groupby("session")["vote"].nunique()
    unanimous = n_options[n_options == 1].index
    valid = valid[~valid["session"].isin(unanimous)]
    print(f"dropped {len(unanimous)} unanimous sessions out of {df.session.nunique()}")

V = valid.pivot(index="actor", columns="session", values="vote")
V = V.reindex(sorted(df["actor"].unique()))     # keep every MP, even one with no valid votes
actors = V.index.to_numpy()
print(V.shape)
V.iloc[:5, :8]

## Agreement rate, vectorised

For each vote option $c$ build the indicator matrix $X^{(c)}_{is} = [v_{is} = c]$. Then

- shared sessions: $N = M M^\top$ with $M_{is} = [v_{is} \text{ valid}]$
- agreements: $K = \sum_c X^{(c)} X^{(c)\top}$

and $A = K / N$. This replaces a double loop over ~19,000 MP pairs with a few matrix products.

In [ ]:
M = V.notna().to_numpy(dtype=float)                       # MP × session, 1 = valid vote
N = M @ M.T                                               # shared valid sessions per pair
K = sum((V == c).to_numpy(dtype=float) @ (V == c).to_numpy(dtype=float).T
        for c in VALID_VOTES)                             # sessions where they voted the same

with np.errstate(invalid="ignore", divide="ignore"):
    A = np.where(N >= MIN_SHARED, K / N, 0.0)
np.fill_diagonal(A, 0.0)

# MPs with no edge left would be trivial one-MP communities in Louvain: remove them
keep = A.sum(axis=1) > 0
print(f"removing {(~keep).sum()} MPs without edges: {actors[~keep].tolist()}")
A, N, M, actors = A[np.ix_(keep, keep)], N[np.ix_(keep, keep)], M[keep], actors[keep]

assert np.allclose(A, A.T) and (A >= 0).all() and (A <= 1).all()
n_pairs = len(A) * (len(A) - 1) // 2
n_dropped = (np.triu(N < MIN_SHARED, k=1)).sum()
print(f"A: {A.shape}, pairs below MIN_SHARED (no edge): {n_dropped} / {n_pairs}")

Sanity check: recompute a few pairs directly from the long-format data.

In [ ]:
rng = np.random.default_rng(0)
for i, j in rng.choice(len(actors), size=(5, 2), replace=True):
    if i == j:
        continue
    a, b = valid[valid.actor == actors[i]], valid[valid.actor == actors[j]]
    both = a.merge(b, on="session")
    rate = (both.vote_x == both.vote_y).mean() if len(both) >= MIN_SHARED else 0.0
    assert np.isclose(rate, A[i, j]), (actors[i], actors[j])
print("direct recomputation matches")

## Party label per MP

A few MPs changed party during the period, so the party is stored only as a label for interpreting communities, not used to build $A$. We keep the party under which the MP cast most votes (`party`) and the last one (`party_last`).

In [ ]:
lab = df.dropna(subset=["party"]).sort_values("date")
party = lab.groupby("actor")["party"].agg(lambda s: s.mode().iloc[0]).reindex(actors).fillna("Unknown")
party_last = lab.groupby("actor")["party"].last().reindex(actors).fillna("Unknown")
n_valid = M.sum(axis=1).astype(int)

print("MPs that switched party:", (lab.groupby("actor")["party"].nunique() > 1).sum())
party.value_counts()

### MPs who switched party

Names are fetched from the API (`Aktør`). The transition chain follows the order of the MP's votes; the date is the first vote cast under the new party.

In [ ]:
import requests

switchers = lab.groupby("actor")["party"].nunique().pipe(lambda s: s[s > 1]).index.tolist()

# Fetch the names of the switchers from the Danish Parliament API
resp = requests.get("https://oda.ft.dk/api/Aktør", params={
    "$format": "json",
    "$select": "id,navn",
    "$filter": " or ".join(f"id eq {a}" for a in switchers),
}, timeout=60).json()
names = {a["id"]: a["navn"] for a in resp["value"]}

for a in switchers:
    votes_a = lab[lab["actor"] == a]
    runs = votes_a[votes_a["party"] != votes_a["party"].shift()]   # first vote under each party
    chain = runs["party"].tolist()
    dates = ", ".join(d.strftime("%Y-%m-%d") for d in runs["date"].iloc[1:])
    print(f"{names.get(a, a)}, {' -> '.join(chain)}  ({dates})")

## Inspect the matrix

Rows/columns ordered by party. Clear blocks along the diagonal mean party discipline is visible in the agreement rates. Note the high baseline: many votes are near-unanimous, so even MPs from opposing parties often agree > 50% of the time.

In [ ]:
order = np.lexsort((actors, party.to_numpy()))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw={"width_ratios": [1.2, 1]})

im = ax1.imshow(A[np.ix_(order, order)], cmap="viridis", vmin=0, vmax=1)
p = party.to_numpy()[order]
edges = np.flatnonzero(p[1:] != p[:-1]) + 1
for e in edges:
    ax1.axhline(e - .5, color="white", lw=.4); ax1.axvline(e - .5, color="white", lw=.4)
ax1.set_title("Agreement rate (MPs sorted by party)")
fig.colorbar(im, ax=ax1, fraction=.046)

same = (party.to_numpy()[:, None] == party.to_numpy()[None, :])
iu = np.triu_indices(len(A), k=1)
w, s = A[iu], same[iu]
ax2.hist(w[s & (w > 0)], bins=50, alpha=.7, density=True, label="same party")
ax2.hist(w[~s & (w > 0)], bins=50, alpha=.7, density=True, label="different party")
ax2.set_title("Pairwise agreement rate"); ax2.set_xlabel("agreement rate"); ax2.legend()
plt.tight_layout()

## Save

`similarity.npz` contains:

| key | shape | description |
|---|---|---|
| `A` | (n, n) | agreement-rate adjacency matrix, symmetric, zero diagonal |
| `N` | (n, n) | number of shared valid sessions per pair |
| `actor` | (n,) | `Aktør.id` of each row/column |
| `party` | (n,) | most frequent party of the MP |
| `party_last` | (n,) | last party of the MP in the period |
| `n_valid` | (n,) | number of valid votes cast by the MP |

Load with `d = np.load("similarity.npz"); A = d["A"]`.

In [ ]:
np.savez(
    OUT_PATH,
    A=A, N=N.astype(int), actor=actors,
    party=party.to_numpy(dtype=str), party_last=party_last.to_numpy(dtype=str),
    n_valid=n_valid,
)
print("saved", OUT_PATH, A.shape)